<a href="https://colab.research.google.com/github/nauzleyabedini/Clinical-Note-Evaluation/blob/main/Clinical_Note_Eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Set Up
1. Load Anthropic, Gemini, and OpenAI APIs from [Github](https://https://github.com/nauzleyabedini/Clinical-Note-Evaluation/blob/main/Setup/clinical_utils.py)
2. Load [ACI-BENCH dataset](https://raw.githubusercontent.com/microsoft/clinical_visit_note_summarization_corpus/refs/heads/main/data/aci-bench/challenge_data/train.csv)
3. Load system prompts from GitHub

In [27]:
# 1. Install dependencies (Colab)
#!pip install -q anthropic google-generativeai

# 2a. Download utility module from GitHub
!wget -q -O clinical_utils.py "https://raw.githubusercontent.com/nauzleyabedini/Clinical-Note-Evaluation/main/Setup/clinical_utils.py?nocache=1"

# 2b. Download prompts modules from GitHub (using cache-busters to bypass GitHub's 5-minute CDN cache)
!wget -q -O note_gen_prompts.py "https://raw.githubusercontent.com/nauzleyabedini/Clinical-Note-Evaluation/main/Prompts/note_gen_prompts.py?nocache=1"
!wget -q -O eval_prompt_current.py "https://raw.githubusercontent.com/nauzleyabedini/Clinical-Note-Evaluation/main/Prompts/eval_prompt_current.py?nocache=1"

# 3. Import your utility modules
import sys
import importlib

# Force Python to forget the old cached versions before importing
for mod in ['clinical_utils', 'note_gen_prompts', 'eval_prompt_current']:
    if mod in sys.modules:
        del sys.modules[mod]

import clinical_utils
import note_gen_prompts
import eval_prompt_current

from google.colab import userdata

# 4. Setup APIs cleanly
anthropic_client, openai_client, gemini_model = clinical_utils.setup_clients(
    userdata.get('AnthropicClinDocEvalAPIKey'),
    userdata.get('OpenAIClinDocEvalAPIKey'),
    userdata.get('Gemini_Clin_Doc_Eval_APIKey')
)

# 5. Load ACI-BENCH dataset cleanly
sample_df = clinical_utils.load_aci_bench()

APIs successfully initialized.
Successfully loaded 67 encounter records from ACI-BENCH.


### Load Dataframes Containing Gold Standard Notes and Gemini-Generated Notes

Use to reduce run-times and API calls.
* **full_df:** Contains all 67 encounters, patient-clinician transcript, gold-standard clinician-generated note ('note'), and Gemini-created AI note.
* **experimental_df:** truncated form of the above that just includes the first 10 encounters.

*Note: Scripts and system prompts for AI note generation from patient-clinician transcripts can be found in [GitHub repository](https://github.com/nauzleyabedini/Clinical-Note-Evaluation/).*

In [28]:
# Load the pre-generated datasets directly from GitHub
import pandas as pd

full_dataset_url = 'https://raw.githubusercontent.com/nauzleyabedini/Clinical-Note-Evaluation/main/Data/full_dataset_with_ai_notes.csv'
experimental_dataset_url = 'https://raw.githubusercontent.com/nauzleyabedini/Clinical-Note-Evaluation/main/Data/experimental_dataset_with_ai_notes.csv'

try:
    full_df = pd.read_csv(full_dataset_url)
    experimental_df = pd.read_csv(experimental_dataset_url)
    print(f"Successfully loaded full dataset with {len(full_df)} encounters from GitHub.")
    print(f"Successfully loaded experimental dataset with {len(experimental_df)} encounters from GitHub.")
    print("\nNote: You can skip the subsequent AI Note Generation cells and proceed directly to Evaluation.")
    display(experimental_df.head(2))
except Exception as e:
    print(f"Error loading datasets from GitHub: {e}")
    print("Please check the URLs or run the generation cells below to create them locally.")


Successfully loaded full dataset with 67 encounters from GitHub.
Successfully loaded experimental dataset with 10 encounters from GitHub.

Note: You can skip the subsequent AI Note Generation cells and proceed directly to Evaluation.


,dataset,encounter_id,dialogue,note,gem_ai_notes_generated,gem_ai_note_post_processed
0,virtassist,D2N001,"[doctor] hi , martha . how are you ?\n[patient...",CHIEF COMPLAINT\n\nAnnual exam.\n\nHISTORY OF ...,CHIEF COMPLAINT\nAnnual physical examination.\...,CHIEF COMPLAINT\nAnnual physical examination.\...
1,virtassist,D2N002,"[doctor] hi , andrew , how are you ?\n[patient...",CHIEF COMPLAINT\n\nJoint pain.\n\nHISTORY OF P...,CHIEF COMPLAINT\nBilateral knee pain.\n\nHISTO...,CHIEF COMPLAINT\nBilateral knee pain.\n\nHISTO...


In [29]:
import google.generativeai as genai
import note_gen_prompts

#-------------------------------------------------------
# PHASE 1: GENERATE GEMINI AI DRAFT NOTES
#-------------------------------------------------------

def gem_ai_note(transcript):
    """
    Uses Gemini to generate an initial AI draft note grounded in the transcript.
    This note is designed to be compliant, accurate, and safe according to outpatient note standards.
    """
    # Load the system prompt dynamically from our external prompts module
    system_prompt = note_gen_prompts.GEMINI_NOTE_GENERATION_PROMPT

    user_prompt = f"""### ENCOUNTER TRANSCRIPT:
{transcript}

Please generate the initial AI draft clinical note now:"""

    # Call Gemini API
    response = gemini_model.generate_content(
        contents=f"{system_prompt}\n\n{user_prompt}"
    )
    return response.text.strip()


In [30]:
# Only run generation if the pre-generated datasets failed to load or are missing
REGENERATE_NOTES = 'full_df' not in globals() or 'experimental_df' not in globals()

if REGENERATE_NOTES:
    gem_ai_notes_generated = []

    print("\n--- Generating Gemini AI Draft Notes ---")
    for i in range(len(sample_df)):
        transcript = sample_df['dialogue'].iloc[i]

        print(f"Generating AI Draft Note for Sample {i+1}...")
        generated_note = gem_ai_note(transcript)
        gem_ai_notes_generated.append(generated_note)

    sample_df['gem_ai_notes_generated'] = gem_ai_notes_generated
    display(sample_df[['encounter_id', 'dialogue', 'note', 'gem_ai_notes_generated']].head())
else:
    print("Pre-loaded datasets (full_df, experimental_df) are available. Skipping note generation.")

Pre-loaded datasets (full_df, experimental_df) are available. Skipping note generation.


### Using LLM-As-A-Judge to Evaluate AI-Generated Notes

Evaluate AI-Generated Notes against Gold-Standard Clinician Notes and/or Patient-Clinician Transcripts where appropriate across multiple domains (broken down into modules). Use both Anthropic and OpenAI APIs for evaluation.

*Note: System prompts for evaluation can be found [here](https://github.com/nauzleyabedini/Clinical-Note-Evaluation/blob/main/Prompts/eval_prompt_current).*

In [42]:
import json
import time
import importlib
import re
from anthropic.types import TextBlock
import eval_prompt_current

# Force reload to ensure we have the absolute latest file from disk
importlib.reload(eval_prompt_current)

# ==============================================================================
# MODULE 1: FACT GROUNDING & CLINICAL SAFETY
# ==============================================================================
# Prompt is loaded dynamically from eval_prompt_current.AUDIT_PROMPT_TEMPLATE_MODULE1

# ==============================================================================
# MODULE 2: CLINICAL REASONING & CONTEXTUAL ATTRIBUTION
# ==============================================================================
# Prompt is loaded dynamically from eval_prompt_current.AUDIT_PROMPT_TEMPLATE_MODULE2

# ==============================================================================
# MODULE 3: CODING, BILLING & COMPLIANCE INTEGRITY
# ==============================================================================
# Prompt is loaded dynamically from eval_prompt_current.AUDIT_PROMPT_TEMPLATE_MODULE3

# ==============================================================================
# MODULE 4: STRUCTURE, USABILITY & MASTER GATE EVALUATION
# ==============================================================================
# Prompt is loaded dynamically from eval_prompt_current.AUDIT_PROMPT_TEMPLATE_MODULE4

def evaluate_with_openai(transcript, gem_ainote, gold_standard_note):
    """Judge 1: OpenAI gpt-4o-mini"""
    all_results = {}

    def call_openai_json(prompt):
        response = openai_client.chat.completions.create(
            model="gpt-4o-mini",
            temperature=0.0,
            response_format={"type": "json_object"},
            messages=[{"role": "user", "content": prompt}]
        )
        return json.loads(response.choices[0].message.content)

    # Module 1
    p1 = eval_prompt_current.AUDIT_PROMPT_TEMPLATE_MODULE1.replace("{transcript}", transcript).replace("{gem_ai_note}", gem_ainote)
    res1 = call_openai_json(p1)
    all_results.update(res1)

    # Module 2
    p2 = eval_prompt_current.AUDIT_PROMPT_TEMPLATE_MODULE2.replace("{transcript}", transcript).replace("{gem_ai_note}", gem_ainote)
    res2 = call_openai_json(p2)
    all_results.update(res2)

    # Module 3
    p3 = eval_prompt_current.AUDIT_PROMPT_TEMPLATE_MODULE3.replace("{transcript}", transcript).replace("{gem_ai_note}", gem_ainote).replace("{gold_standard_note}", gold_standard_note)
    res3 = call_openai_json(p3)
    all_results.update(res3)

    # Module 4
    p4 = eval_prompt_current.AUDIT_PROMPT_TEMPLATE_MODULE4.replace("{gem_ai_note}", gem_ainote).replace("{module_1_json_output}", json.dumps(res1)).replace("{module_2_json_output}", json.dumps(res2)).replace("{module_3_json_output}", json.dumps(res3))
    res4 = call_openai_json(p4)
    all_results.update(res4)

    return all_results

def evaluate_with_anthropic(transcript, gem_ainote, gold_standard_note):
    """Judge 2: Anthropic claude-sonnet-5"""
    all_results = {}
    max_retries = 5

    def call_anthropic_module(prompt_template, module_name, **kwargs):
        prompt = prompt_template
        # Safely inject variables using replace instead of format to avoid json curly brace errors
        for k, v in kwargs.items():
            prompt = prompt.replace(f"{{{k}}}", str(v))

        for i in range(max_retries):
            try:
                response = anthropic_client.messages.create(
                    model="claude-sonnet-5",
                    max_tokens=8192,
                    messages=[{"role": "user", "content": prompt}]
                )
                raw_text = ""
                for block in response.content:
                    if isinstance(block, TextBlock):
                        raw_text = block.text
                        break

                # Robust JSON extraction
                clean_json = raw_text.replace("```json", "").replace("```", "").strip()
                start_idx = clean_json.find('{')
                end_idx = clean_json.rfind('}')
                if start_idx != -1 and end_idx != -1 and end_idx > start_idx:
                    clean_json = clean_json[start_idx:end_idx+1]

                return json.loads(clean_json)
            except Exception as e:
                if i < max_retries - 1:
                    print(f"Anthropic API error for {module_name}: {e}. Retrying...")
                    time.sleep(2 ** i)
                else:
                    raise
        return {}

    # Module 1
    res1 = call_anthropic_module(eval_prompt_current.AUDIT_PROMPT_TEMPLATE_MODULE1, "Module 1", transcript=transcript, gem_ai_note=gem_ainote)
    all_results.update(res1)

    # Module 2
    res2 = call_anthropic_module(eval_prompt_current.AUDIT_PROMPT_TEMPLATE_MODULE2, "Module 2", transcript=transcript, gem_ai_note=gem_ainote)
    all_results.update(res2)

    # Module 3
    res3 = call_anthropic_module(eval_prompt_current.AUDIT_PROMPT_TEMPLATE_MODULE3, "Module 3", transcript=transcript, gem_ai_note=gem_ainote, gold_standard_note=gold_standard_note)
    all_results.update(res3)

    # Module 4
    res4 = call_anthropic_module(eval_prompt_current.AUDIT_PROMPT_TEMPLATE_MODULE4, "Module 4",
                                 gem_ai_note=gem_ainote,
                                 module_1_json_output=json.dumps(res1),
                                 module_2_json_output=json.dumps(res2),
                                 module_3_json_output=json.dumps(res3))
    all_results.update(res4)

    return all_results

In [34]:
import pandas as pd
import json
from datetime import datetime

# Define a version tag to keep track of different evaluation runs/models
version_tag = "v2_4_modules" # Change this to your preferred version name (e.g., 'v3_new_prompts')

# Ensure datasets are loaded (fallback if the GitHub cells weren't run)
if 'full_df' not in globals() or 'experimental_df' not in globals():
    print("Loading datasets locally as fallback...")
    full_df = pd.read_csv('full_dataset_with_ai_notes.csv')
    experimental_df = pd.read_csv('experimental_dataset_with_ai_notes.csv')

def run_evaluation(df, dataset_name):
    all_eval_results = []
    print(f"\n--- Evaluating {len(df)} sample notes from {dataset_name} ---")
    for i in range(len(df)):
        transcript_to_evaluate = df['dialogue'].iloc[i]
        # Use pre-generated if available, fallback to 'note'
        ai_note_to_evaluate = df['gem_ai_note_post_processed'].iloc[i] if 'gem_ai_note_post_processed' in df.columns else df['note'].iloc[i]
        gold_standard_note_for_eval = df['note'].iloc[i]

        print(f"\n--- Sample Note {i+1} ({dataset_name}) ---")

        # Evaluate with OpenAI
        print("OpenAI Evaluation:")
        try:
            openai_evaluation_result = evaluate_with_openai(transcript_to_evaluate, ai_note_to_evaluate, gold_standard_note_for_eval)
            openai_evaluation_result['model'] = 'OpenAI'
            openai_evaluation_result['sample_id'] = i + 1
            openai_evaluation_result['dataset'] = dataset_name
            all_eval_results.append(openai_evaluation_result)
            print("OpenAI evaluation successful.")
        except Exception as e:
            print(f"Error evaluating sample {i+1} with OpenAI: {e}")

        # Evaluate with Anthropic
        print("Anthropic Evaluation:")
        try:
            anthropic_evaluation_result = evaluate_with_anthropic(transcript_to_evaluate, ai_note_to_evaluate, gold_standard_note_for_eval)
            anthropic_evaluation_result['model'] = 'Anthropic'
            anthropic_evaluation_result['sample_id'] = i + 1
            anthropic_evaluation_result['dataset'] = dataset_name
            all_eval_results.append(anthropic_evaluation_result)
            print("Anthropic evaluation successful.")
        except Exception as e:
            print(f"Error evaluating sample {i+1} with Anthropic: {e}")

    print(f"\n--- Aggregating results for {dataset_name} ---")
    flattened_results = []
    for res in all_eval_results:
        # Start with base attributes
        flattened_res = {
            'dataset': res.get('dataset'),
            'model': res.get('model'),
            'sample_id': res.get('sample_id')
        }

        # Flatten nested JSON structures for tabular saving
        for module_name, module_data in res.items():
            if isinstance(module_data, dict) and module_name.startswith('module_'):
                for key, value in module_data.items():
                    if isinstance(value, (list, dict)):
                        # Serialize complex nested structures (like quote lists) to JSON strings
                        flattened_res[key] = json.dumps(value)
                    else:
                        flattened_res[key] = value

        flattened_results.append(flattened_res)

    return pd.DataFrame(flattened_results)

# Run evaluations on datasets (limit experimental to speed up testing if needed)
aggregated_exp_df = run_evaluation(experimental_df, 'Experimental Dataset')

print("\n--- Experimental Dataset Evaluation Results ---")
display(aggregated_exp_df.head())

# Save aggregated results to CSV with version tag
exp_filename = f'aggregated_experimental_evaluation_results_{version_tag}.csv'
aggregated_exp_df.to_csv(exp_filename, index=False)
print(f"\nSaved experimental evaluation results to '{exp_filename}'")

print("\nNow running full dataset evaluation...")
aggregated_full_df = run_evaluation(full_df, 'Full Dataset')
full_filename = f'aggregated_full_evaluation_results_{version_tag}.csv'
aggregated_full_df.to_csv(full_filename, index=False)
print(f"Saved full evaluation results to '{full_filename}'")


--- Evaluating 10 sample notes from Experimental Dataset ---

--- Sample Note 1 (Experimental Dataset) ---
OpenAI Evaluation:
OpenAI evaluation successful.
Anthropic Evaluation:
Anthropic evaluation successful.

--- Sample Note 2 (Experimental Dataset) ---
OpenAI Evaluation:
OpenAI evaluation successful.
Anthropic Evaluation:
Anthropic evaluation successful.

--- Sample Note 3 (Experimental Dataset) ---
OpenAI Evaluation:
OpenAI evaluation successful.
Anthropic Evaluation:
Anthropic evaluation successful.

--- Sample Note 4 (Experimental Dataset) ---
OpenAI Evaluation:
OpenAI evaluation successful.
Anthropic Evaluation:
Anthropic evaluation successful.

--- Sample Note 5 (Experimental Dataset) ---
OpenAI Evaluation:
OpenAI evaluation successful.
Anthropic Evaluation:
Anthropic evaluation successful.

--- Sample Note 6 (Experimental Dataset) ---
OpenAI Evaluation:
OpenAI evaluation successful.
Anthropic Evaluation:
Anthropic evaluation successful.

--- Sample Note 7 (Experimental Datas

,dataset,model,sample_id,extracted_hallucinations,extracted_omissions,extracted_contradictions,contains_critical_safety_error,extracted_misattributions,contains_critical_reasoning_error,hcc_meat_audit,unsubstantiated_diagnoses,icd10_specificity_score,contains_critical_compliance_error,structure_usability_score,overall_note_pass,failing_triggers,executive_summary
0,Experimental Dataset,OpenAI,1,[],"[{""transcript_quote"": ""i'm still forgetting to...",[],True,"[{""note_quote"": ""She denies chest pain or shor...",False,"{""total_active_chronic_conditions"": 3, ""condit...",[],2,False,2,False,"[""Critical_Safety_Error""]",The note is generally well-structured and read...
1,Experimental Dataset,Anthropic,1,"[{""note_quote"": ""Lungs: Clear/normal exam."", ""...","[{""transcript_quote"": ""i've been traveling a l...",[],False,"[{""note_quote"": ""Treat: Increase Lisinopril to...",False,"{""total_active_chronic_conditions"": 3, ""condit...","[{""diagnosed_condition"": ""Congestive Heart Fai...",2,False,2,True,[],The note is well-organized with clear section ...
2,Experimental Dataset,OpenAI,2,"[{""note_quote"": ""Regarding his hypothyroidism,...","[{""transcript_quote"": ""so other than your knee...","[{""note_quote"": ""Acute exacerbation of osteoar...",False,"[{""note_quote"": ""The provider verbally queried...",True,"{""total_active_chronic_conditions"": 3, ""condit...",[],3,False,2,False,"[""Critical_Reasoning_Error""]",The note is generally well-structured and read...
3,Experimental Dataset,Anthropic,2,"[{""note_quote"": ""M17.11 - Primary osteoarthrit...",[],[],False,"[{""note_quote"": ""[CLINICIAN REVIEW FLAG: The p...",False,"{""total_active_chronic_conditions"": 3, ""condit...","[{""diagnosed_condition"": ""Primary osteoarthrit...",1,True,2,False,"[""Critical_Compliance_Error"", ""Unsubstantiated...",The note is well-organized with clear SOAP-sty...
4,Experimental Dataset,OpenAI,3,[],"[{""transcript_quote"": ""i think i do . it's kin...","[{""note_quote"": ""Positive for right CVA tender...",False,"[{""note_quote"": ""Patient reported that back pa...",True,"{""total_active_chronic_conditions"": 3, ""condit...",[],3,False,2,False,"[""Critical_Reasoning_Error""]",The note is generally well-structured and read...



Saved experimental evaluation results to 'aggregated_experimental_evaluation_results_v2_4_modules.csv'

Now running full dataset evaluation...

--- Evaluating 67 sample notes from Full Dataset ---

--- Sample Note 1 (Full Dataset) ---
OpenAI Evaluation:
OpenAI evaluation successful.
Anthropic Evaluation:
Anthropic evaluation successful.

--- Sample Note 2 (Full Dataset) ---
OpenAI Evaluation:
OpenAI evaluation successful.
Anthropic Evaluation:
Anthropic evaluation successful.

--- Sample Note 3 (Full Dataset) ---
OpenAI Evaluation:
OpenAI evaluation successful.
Anthropic Evaluation:
Anthropic evaluation successful.

--- Sample Note 4 (Full Dataset) ---
OpenAI Evaluation:
OpenAI evaluation successful.
Anthropic Evaluation:
Anthropic evaluation successful.

--- Sample Note 5 (Full Dataset) ---
OpenAI Evaluation:
OpenAI evaluation successful.
Anthropic Evaluation:
Anthropic evaluation successful.

--- Sample Note 6 (Full Dataset) ---
OpenAI Evaluation:
OpenAI evaluation successful.
Anthr

In [35]:
print(aggregated_exp_df.to_markdown(index=False))
print(aggregated_full_df.to_markdown(index=False))

| dataset              | model     |   sample_id | extracted_hallucinations                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             

### Load Pre-Evaluated Results from GitHub

Run this cell to load the already evaluated datasets directly from your GitHub repository. This saves you from having to re-run the expensive and time-consuming LLM API calls above every time you restart the notebook.

In [54]:
import pandas as pd

# URLs to your pre-evaluated datasets hosted on GitHub
# Update these if your file names or folder structures change
exp_eval_url = 'https://raw.githubusercontent.com/nauzleyabedini/Clinical-Note-Evaluation/main/Data/aggregated_experimental_evaluation_results_v2_4_modules.csv'
full_eval_url = 'https://raw.githubusercontent.com/nauzleyabedini/Clinical-Note-Evaluation/main/Data/aggregated_full_evaluation_results_v2_4_modules.csv'

print("Attempting to load pre-evaluated datasets from GitHub to save time...\n")

try:
    # We overwrite the aggregated_exp_df variable with the one from GitHub
    aggregated_exp_df = pd.read_csv(exp_eval_url)
    print(f"✅ Successfully loaded experimental evaluation results ({len(aggregated_exp_df)} rows).")
except Exception as e:
    print(f"⚠️ Could not load experimental evaluation results from GitHub. (Have you pushed it to the repo yet?)\nError: {e}\n")

try:
    # We overwrite the aggregated_full_df variable with the one from GitHub
    aggregated_full_df = pd.read_csv(full_eval_url)
    print(f"✅ Successfully loaded full evaluation results ({len(aggregated_full_df)} rows).")
except Exception as e:
    print(f"⚠️ Could not load full evaluation results from GitHub. (Have you pushed it to the repo yet?)\nError: {e}")

# Quick check to show it loaded correctly
if 'aggregated_exp_df' in globals() and not aggregated_exp_df.empty:
    print("\nPreview of loaded data:")
    display(aggregated_exp_df.head(2))

Attempting to load pre-evaluated datasets from GitHub to save time...

✅ Successfully loaded experimental evaluation results (20 rows).
✅ Successfully loaded full evaluation results (134 rows).

Preview of loaded data:


,dataset,model,sample_id,extracted_hallucinations,extracted_omissions,extracted_contradictions,contains_critical_safety_error,extracted_misattributions,contains_critical_reasoning_error,hcc_meat_audit,unsubstantiated_diagnoses,icd10_specificity_score,contains_critical_compliance_error,structure_usability_score,overall_note_pass,failing_triggers,executive_summary
0,Experimental Dataset,OpenAI,1,[],"[{""transcript_quote"": ""i'm still forgetting to...",[],True,"[{""note_quote"": ""She denies chest pain or shor...",False,"{""total_active_chronic_conditions"": 3, ""condit...",[],2,False,2,False,"[""Critical_Safety_Error""]",The note is generally well-structured and read...
1,Experimental Dataset,Anthropic,1,"[{""note_quote"": ""Lungs: Clear/normal exam."", ""...","[{""transcript_quote"": ""i've been traveling a l...",[],False,"[{""note_quote"": ""Treat: Increase Lisinopril to...",False,"{""total_active_chronic_conditions"": 3, ""condit...","[{""diagnosed_condition"": ""Congestive Heart Fai...",2,False,2,True,[],The note is well-organized with clear section ...


### Evaluation Results Summary

This section summarizes the main findings of the evaluation, comparing how many notes were judged as failing by each LLM and the primary reasons (failing triggers) for those failures.

In [55]:
import pandas as pd
import json

def standardize_trigger(raw_trigger):
    """
    Maps verbose or specific LLM triggers back to the core rubric categories.
    """
    trigger_str = str(raw_trigger).lower().replace(" ", "_")

    # Official core categories mapping
    if 'safety_error' in trigger_str or 'fabricat' in trigger_str or 'hallucin' in trigger_str:
        return 'Critical_Safety_Error'
    if 'safety_omission' in trigger_str or 'omission_critical' in trigger_str or 'omis' in trigger_str:
        return 'Critical_Safety_Omission'
    if 'compliance_error' in trigger_str or 'hcc_meat_fail' in trigger_str:
        return 'Critical_Compliance_Error'
    if 'reasoning' in trigger_str or 'contradiction' in trigger_str or 'misattribution' in trigger_str or 'timeline_displacement' in trigger_str:
        return 'Critical_Reasoning_Error'
    if 'unsubstantiated' in trigger_str:
        return 'Unsubstantiated_Diagnosis'
    if 'upcoding' in trigger_str:
        return 'Upcoding_Risk'
    if 'misplacement' in trigger_str or 'structure' in trigger_str:
        return 'Note_Structure_Error'
    if 'compliance' in trigger_str:
        return 'Critical_Compliance_Error'
    if 'safety' in trigger_str:
        return 'Critical_Safety_Error'

    # Fallback if we can't map it
    return 'Other_Rubric_Violation'

def summarize_failures(df, dataset_name):
    print(f"========== Failure Summary for {dataset_name} ==========")
    if df is None or df.empty:
        print("No data available.\n")
        return

    if 'overall_note_pass' not in df.columns:
        print("Column 'overall_note_pass' not found. Ensure Module 4 completed successfully.\n")
        return

    for model in df['model'].unique():
        model_df = df[df['model'] == model]
        total_notes = len(model_df)

        # Find failures (overall_note_pass == False)
        fails = model_df[(model_df['overall_note_pass'] == False) | (model_df['overall_note_pass'] == 'False')]
        fail_count = len(fails)

        print(f"\n🤖 Judge Model: {model}")
        print(f"Total Notes Evaluated: {total_notes}")
        print(f"Total Failed Notes: {fail_count} ({(fail_count/total_notes)*100:.1f}% Fail Rate)")

        if fail_count > 0 and 'failing_triggers' in df.columns:
            print("Main Reasons for Failure (Standardized to Rubric):")
            reasons = []
            for triggers in fails['failing_triggers'].dropna():
                try:
                    parsed_triggers = json.loads(triggers)
                    if isinstance(parsed_triggers, list):
                        for pt in parsed_triggers:
                            reasons.append(standardize_trigger(pt))
                    else:
                        reasons.append(standardize_trigger(parsed_triggers))
                except:
                    reasons.append(standardize_trigger(triggers))

            if reasons:
                # Count frequencies of each standardized reason
                reason_counts = pd.Series(reasons).value_counts()
                for reason, count in reason_counts.items():
                    print(f"  - {reason}: {count} occurrences")
            else:
                print("  - No specific triggers extracted.")
        elif fail_count > 0:
             print("  - 'failing_triggers' column not found to explain failures.")
    print("\n")

# Run the summary on the experimental dataset
if 'aggregated_exp_df' in globals():
    summarize_failures(aggregated_exp_df, "Experimental Dataset")

# Run the summary on the full dataset if it completed
if 'aggregated_full_df' in globals() and not aggregated_full_df.empty:
    summarize_failures(aggregated_full_df, "Full Dataset")
else:
    print("Note: Full dataset summary skipped because 'aggregated_full_df' is empty or not yet fully evaluated.")

========== Failure Summary for Experimental Dataset ==========

🤖 Judge Model: OpenAI
Total Notes Evaluated: 10
Total Failed Notes: 6 (60.0% Fail Rate)
Main Reasons for Failure (Standardized to Rubric):
  - Critical_Reasoning_Error: 3 occurrences
  - Critical_Safety_Error: 1 occurrences
  - Critical_Safety_Omission: 1 occurrences
  - Critical_Compliance_Error: 1 occurrences

🤖 Judge Model: Anthropic
Total Notes Evaluated: 10
Total Failed Notes: 5 (50.0% Fail Rate)
Main Reasons for Failure (Standardized to Rubric):
  - Unsubstantiated_Diagnosis: 4 occurrences
  - Critical_Compliance_Error: 3 occurrences
  - Critical_Safety_Error: 2 occurrences
  - Note_Structure_Error: 1 occurrences
  - Critical_Reasoning_Error: 1 occurrences


========== Failure Summary for Full Dataset ==========

🤖 Judge Model: Anthropic
Total Notes Evaluated: 67
Total Failed Notes: 47 (70.1% Fail Rate)
Main Reasons for Failure (Standardized to Rubric):
  - Critical_Compliance_Error: 32 occurrences
  - Unsubstantiate

## Visualizing LLM-as-a-Judge Results


## Human-in-the-Loop Evaluation

To compare human expert evaluation against the LLM-as-a-Judge, we need to record human responses in the exact same format.

Run the cell below to generate and download a blank CSV template. It contains the exact metrics expected by the data visualizations above. Once you fill it out (e.g., in Excel or Google Sheets), you can upload it back to Colab and merge it with the main dataframe.

In [57]:
import pandas as pd
from google.colab import files

# Dynamically extract all relevant metric columns from the latest evaluation results
# We exclude metadata and verbose text/JSON columns so the human only fills out the scores/booleans
exclude_cols = [
    'dataset', 'model', 'sample_id',
    'extracted_hallucinations', 'extracted_omissions', 'extracted_contradictions',
    'extracted_misattributions', 'hcc_meat_audit', 'unsubstantiated_diagnoses',
    'failing_triggers', 'overall_key_findings', 'executive_summary'
]

if 'aggregated_exp_df' in globals():
    metrics_to_evaluate = [col for col in aggregated_exp_df.columns if col not in exclude_cols]
else:
    # Fallback to manual list if the dataframe isn't loaded
    metrics_to_evaluate = [
        'transcript_fidelity_score',
        'hallucinations_severity_score',
        'critical_omissions_score',
        'contains_critical_safety_error',
        'contains_critical_reasoning_error',
        'icd10_specificity_score',
        'contains_critical_compliance_error',
        'clinical_validation_score',
        'note_structure_organization_score',
        'safety_risk_tier_score',
        'overall_note_pass'
    ]

# Create a template dataframe for the 10 experimental samples
human_eval_data = []
for i in range(1, 11):
    row = {
        'dataset': 'Experimental Dataset',
        'model': 'Human', # This will act as our 3rd 'judge' model in graphs
        'sample_id': i
    }
    # Create empty columns for the human to fill in
    for metric in metrics_to_evaluate:
        row[metric] = ''
    human_eval_data.append(row)

human_eval_df = pd.DataFrame(human_eval_data)

# Save to CSV
template_filename = 'human_evaluation_template.csv'
human_eval_df.to_csv(template_filename, index=False)

print(f"Created template: {template_filename}")
print("Downloading file to your computer...")

# Preview the template
display(human_eval_df.head())

# Trigger the download
files.download(template_filename)


Created template: human_evaluation_template.csv


,dataset,model,sample_id,contains_critical_safety_error,contains_critical_reasoning_error,icd10_specificity_score,contains_critical_compliance_error,structure_usability_score,overall_note_pass
0,Experimental Dataset,Human,1,,,,,,
1,Experimental Dataset,Human,2,,,,,,
2,Experimental Dataset,Human,3,,,,,,
3,Experimental Dataset,Human,4,,,,,,
4,Experimental Dataset,Human,5,,,,,,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Merging Human Data (Run after filling out the template)

Once you have filled out the CSV and uploaded it back to the Colab files pane, run the cell below to merge it with your LLM results. You can then re-run the graphing cells above to see 'Human' alongside 'OpenAI' and 'Anthropic'.

In [ ]:
import os

# Check if the user uploaded the completed file
if os.path.exists('human_evaluation_template.csv'):
    try:
        # Load the completed human evaluation
        completed_human_df = pd.read_csv('human_evaluation_template.csv')

        # Optional: Drop rows that weren't filled out yet just in case
        completed_human_df = completed_human_df.dropna(subset=['transcript_fidelity_score'])

        if not completed_human_df.empty:
            # Merge with the existing aggregated_exp_df
            # We use pd.concat to add the human rows to the bottom
            aggregated_df = pd.concat([aggregated_exp_df, completed_human_df], ignore_index=True)
            print("✅ Successfully merged Human evaluation data with LLM evaluation data!")
            print("You can now scroll up and re-run the visualization cells (Bar charts, Heatmaps, etc.) to compare.")
            display(aggregated_df[aggregated_df['model'] == 'Human'].head())
        else:
            print("⚠️ The loaded CSV appears to be empty or incomplete.")
    except Exception as e:
        print(f"Error loading human data: {e}")
else:
    print("⚠️ 'human_evaluation_template.csv' not found. Please upload your completed file to the Colab files pane first.")

### Human vs. AI Agreement (Cohen's Kappa)

Run this cell **after** merging your completed `human_evaluation_template.csv` above. This will calculate the Inter-Rater Reliability (Cohen's Kappa) comparing your human expert scores against both OpenAI and Anthropic to see which model aligns closer to human clinical judgment.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import cohen_kappa_score

if 'aggregated_df' in globals() and 'Human' in aggregated_df['model'].values:
    print("========== Human vs. LLM Agreement (Cohen's Kappa) ==========")
    print("Interpretation: <0 (No agreement), 0.0-0.20 (Slight), 0.21-0.40 (Fair), 0.41-0.60 (Moderate), 0.61-0.80 (Substantial), 0.81-1.00 (Almost perfect)\n")

    # Metrics suitable for Kappa (ordinal/categorical/boolean scoring)
    metrics_to_compare = [
        'contains_critical_safety_error',
        'contains_critical_reasoning_error',
        'icd10_specificity_score',
        'contains_critical_compliance_error',
        'structure_usability_score',
        'overall_note_pass'
    ]

    for metric in metrics_to_compare:
        metric_display_name = metric.replace('_score', '').replace('_', ' ').title()
        print(f"--- {metric_display_name} ---")

        try:
            # Pivot table to align sample_ids perfectly across all 3 judges
            pivot_df = aggregated_df.pivot(index='sample_id', columns='model', values=metric)

            # Ensure all three judges are present
            if not all(judge in pivot_df.columns for judge in ['Human', 'OpenAI', 'Anthropic']):
                 print("  Missing one or more judges for this metric.\n")
                 continue

            # Convert all to strings to uniformly handle True/False booleans and numeric scores
            pivot_df = pivot_df[['Human', 'OpenAI', 'Anthropic']].astype(str)
            # Replace empty strings or 'nan' string representations with actual NaN to drop them
            pivot_df = pivot_df.replace(['', 'nan', 'NaN', 'None', '<NA>'], np.nan).dropna()

            if len(pivot_df) == 0:
                print("  Not enough complete/valid data to compare this metric.\n")
                continue

            h_scores = pivot_df['Human']
            o_scores = pivot_df['OpenAI']
            a_scores = pivot_df['Anthropic']

            # Human vs OpenAI
            kappa_o = cohen_kappa_score(h_scores, o_scores)
            # Human vs Anthropic
            kappa_a = cohen_kappa_score(h_scores, a_scores)

            # Handle edge cases where variance is zero (returns NaN)
            str_o = f"{kappa_o:.2f}" if not pd.isna(kappa_o) else "NaN (No variance in scores)"
            str_a = f"{kappa_a:.2f}" if not pd.isna(kappa_a) else "NaN (No variance in scores)"

            print(f"  Human vs OpenAI:    {str_o}")
            print(f"  Human vs Anthropic: {str_a}\n")

        except Exception as e:
            print(f"  Error calculating agreement for {metric}: {e}\n")
else:
    print("⚠️ 'Human' evaluation data not yet merged into 'aggregated_df'. Please upload the CSV and run the merge cell above first.")


### Visualizing Human vs. AI Performance

Run this cell after merging your human data. This generates publication-quality charts perfect for your GitHub README to showcase the concordance between human clinical expertise and LLM evaluations.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from sklearn.metrics import cohen_kappa_score

if 'aggregated_df' in globals() and 'Human' in aggregated_df['model'].values:
    print("Generating Human vs AI Visualizations...")

    sns.set_theme(style="whitegrid")
    metrics_to_plot = [
        'contains_critical_safety_error',
        'contains_critical_reasoning_error',
        'icd10_specificity_score',
        'contains_critical_compliance_error',
        'structure_usability_score',
        'overall_note_pass'
    ]

    # ---------------------------------------------------------
    # Plot 1: Average Scores/Rates Comparison (Grouped Bar Chart)
    # ---------------------------------------------------------
    plt.figure(figsize=(14, 6))

    # Filter and melt dataframe for plotting
    subset_df = aggregated_df[['model'] + metrics_to_plot].copy()

    # Convert booleans to 1/0 for averaging
    for m in metrics_to_plot:
        subset_df[m] = subset_df[m].astype(str).replace({'True': 1, 'False': 0, 'true': 1, 'false': 0})
        subset_df[m] = pd.to_numeric(subset_df[m], errors='coerce')

    melted_df = pd.melt(subset_df, id_vars=['model'], value_vars=metrics_to_plot, var_name='Metric', value_name='Score')
    melted_df['Metric'] = melted_df['Metric'].str.replace('_score', '').str.replace('_', ' ').str.title()

    # Ensure Human is plotted first/consistently by setting category order
    model_order = ['Human', 'OpenAI', 'Anthropic']

    ax = sns.barplot(data=melted_df, x='Metric', y='Score', hue='model', hue_order=model_order, palette=['#2ca02c', '#1f77b4', '#ff7f0e'], errorbar=None)
    plt.title('Average Evaluation Scores & Error Rates: Human Expert vs LLM Judges', fontsize=16, fontweight='bold')
    plt.xlabel('Evaluation Metric', fontsize=12)
    plt.ylabel('Average Score / Rate (0 to 1 for Booleans)', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.legend(title='Judge')
    plt.tight_layout()
    plt.savefig('human_vs_llm_averages.png', dpi=300)
    plt.show()

    # ---------------------------------------------------------
    # Plot 2: Inter-Rater Reliability (Kappa) Heatmap
    # ---------------------------------------------------------
    kappa_results = {'Metric': [], 'Human vs OpenAI': [], 'Human vs Anthropic': []}

    for metric in metrics_to_plot:
        pivot_df = aggregated_df.pivot(index='sample_id', columns='model', values=metric)
        if not all(judge in pivot_df.columns for judge in ['Human', 'OpenAI', 'Anthropic']):
             continue

        # Convert all to strings and handle NaNs consistently
        pivot_df = pivot_df[['Human', 'OpenAI', 'Anthropic']].astype(str)
        pivot_df = pivot_df.replace(['', 'nan', 'NaN', 'None', '<NA>'], np.nan).dropna()

        if len(pivot_df) > 0:
            h_scores = pivot_df['Human']
            o_scores = pivot_df['OpenAI']
            a_scores = pivot_df['Anthropic']

            ko = cohen_kappa_score(h_scores, o_scores)
            ka = cohen_kappa_score(h_scores, a_scores)

            kappa_results['Metric'].append(metric.replace('_score', '').replace('_', ' ').title())
            # Use 0 for NaN (no variance edge case) just for visualization purposes
            kappa_results['Human vs OpenAI'].append(ko if not pd.isna(ko) else 0.0)
            kappa_results['Human vs Anthropic'].append(ka if not pd.isna(ka) else 0.0)

    kappa_df = pd.DataFrame(kappa_results).set_index('Metric')

    plt.figure(figsize=(8, 6))
    # Using a divergent colormap: Red = poor agreement, Green = strong agreement
    sns.heatmap(kappa_df, annot=True, cmap='RdYlGn', vmin=-0.5, vmax=1.0, center=0, fmt='.2f',
                cbar_kws={'label': "Cohen's Kappa"})
    plt.title("Inter-Rater Reliability (Cohen's Kappa)\nAgreement with Human Expert", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('human_agreement_heatmap.png', dpi=300)
    plt.show()

    print("\n✅ Visualizations saved to the Colab files pane as 'human_vs_llm_averages.png' and 'human_agreement_heatmap.png'")
    print("You can download these and add them to your GitHub README!")
else:
    print("⚠️ 'Human' data not found in 'aggregated_df'. Please upload and merge the human template first.")


# AI-Powered Clinical Documentation Evaluation Harness

This project presents a robust harness for the comprehensive evaluation of AI-generated clinical notes against established quality, accuracy, and compliance standards. Leveraging advanced Large Language Models (LLMs) and quantitative metrics, this system addresses critical challenges in healthcare AI, such as hallucination detection, clinical accuracy, and billing compliance.

## 🚀 Quick Start & Resources

To run this pipeline or adapt it for your own use, all necessary scripts and datasets are hosted in the [Clinical-Note-Evaluation GitHub Repository](https://github.com/nauzleyabedini/Clinical-Note-Evaluation/).

**Data Files:**
* [ACI-BENCH Raw Dataset (Train)](https://raw.githubusercontent.com/microsoft/clinical_visit_note_summarization_corpus/refs/heads/main/data/aci-bench/challenge_data/train.csv)
* [Pre-generated Experimental AI Notes (10 samples)](https://raw.githubusercontent.com/nauzleyabedini/Clinical-Note-Evaluation/main/Data/experimental_dataset_with_ai_notes.csv)
* [Pre-generated Full AI Notes (67 samples)](https://raw.githubusercontent.com/nauzleyabedini/Clinical-Note-Evaluation/main/Data/full_dataset_with_ai_notes.csv)

**Prompts & Utilities:**
* `clinical_utils.py`: [Setup and API Handling](https://github.com/nauzleyabedini/Clinical-Note-Evaluation/blob/main/Setup/clinical_utils.py)
* `note_gen_prompts.py`: [Gemini Note Draft Generation Prompts](https://github.com/nauzleyabedini/Clinical-Note-Evaluation/blob/main/Prompts/note_gen_prompts.py)
* `eval_prompt_current.py`: [LLM-as-a-Judge Rubrics](https://github.com/nauzleyabedini/Clinical-Note-Evaluation/blob/main/Prompts/eval_prompt_current.py)


---

## 💡 Project Overview

The harness is designed to meticulously audit AI-generated clinical documentation. It integrates multiple frontier LLMs to simulate real-world auditing scenarios and measure the efficiency gains and potential burdens introduced by AI in clinical settings.

### Why Claude vs. GPT-4o-mini?
We explicitly chose to compare a highly capable, larger model (Anthropic's Claude) against a smaller, highly cost-effective model (OpenAI's GPT-4o-mini). This architectural contrast serves two purposes:
1. **Clear Discernment:** To clearly discern and quantify the differences in clinical reasoning, safety netting, and strictness between different tiers of AI models.
2. **Cost-Optimization Pipeline:** To establish a baseline capability gap, with the ultimate goal of determining if techniques like Automatic Prompt Optimization (APO) can enhance the smaller, cheaper model to operate at the same clinical accuracy level as the more expensive Anthropic model.

## 🔬 Methodology

### AI Note Generation
The Gemini API and a rigorously refined system prompt are leveraged to generate AI clinical notes from outpatient patient-clinician transcripts derived from the [ACI-BENCH dataset](https://arxiv.org/abs/2306.02022).

### AI Note Evaluation (LLM-as-a-Judge)
The quality of AI-generated draft notes is compared against the original transcripts and a "Gold Standard Note," simulating an expert Clinical Documentation Integrity (CDI) audit. The evaluation is broken into four strict modules:
1. **Fact Grounding & Clinical Safety:** Transcript fidelity, hallucination extraction, and critical omissions.
2. **Clinical Reasoning & Contextual Attribution:** Contextual accuracy and timeline integrity.
3. **Coding, Billing & Compliance Integrity:** HCC/MEAT criteria validation, ICD-10 specificity, and upcoding risk assessment.
4. **Structure, Usability & Master Gate Evaluation:** Adherence to standard outpatient note formats (SOAP) and an overall Pass/Fail gate.
The modules and rating scales are described in more detail in `Note_Audit_Rubric_v2.md`: [Initial Evaluation Rubric](https://github.com/nauzleyabedini/Clinical-Note-Evaluation/blob/main/Rubrics/Note_Audit_Rubric_v2.md)

## 📊 Results

### Initial LLM-as-a-Judge Evaluation (Full Dataset - 67 encounters)
Our baseline evaluation highlights significant differences in strictness and clinical auditing capabilities between the two judge models:

* **Anthropic (Claude-Sonnet-5) - The Strict Auditor:** Exhibited a highly conservative profile with a **70.1% failure rate** (47/67 notes failed). It heavily penalized notes for `Critical Compliance Errors` (32 occurrences) and `Unsubstantiated Diagnoses` (26 occurrences), catching fine-grained nuances in the transcript.
* **OpenAI (GPT-4o-mini) - The Lenient Judge:** Evaluated notes much more loosely, yielding a **46.3% failure rate** (31/67 notes failed). While it still caught major `Critical Compliance Errors` (21 occurrences), it missed the vast majority of `Unsubstantiated Diagnoses` (only 7 occurrences flagged) compared to Anthropic.

These findings mathematically demonstrate that relying solely on a smaller, unoptimized model for clinical auditing risks letting clinically significant and compliance-related errors slip into the final medical record. This establishes the baseline capability gap needed for the next phase of prompt optimization.

## 🚀 Next Steps

The next phase of this project focuses on directly bridging the gap between LLM evaluation and human clinical expertise, and optimizing the cost-efficiency of the pipeline:

1. **Complete Human-in-the-Loop Evaluation:** Finalize the manual human expert evaluation on the 10-note experimental subset using the generated CSV rubrics.
2. **Human vs. LLM Benchmarking:** Statistically compare the human expert scores against both Anthropic and OpenAI. This will identify exactly where the LLM judges fail to capture true clinical nuance and which model better aligns with actual physician standards.
3. **Automatic Prompt Optimization (APO):** Introduce APO to iteratively refine and self-improve the LLM-as-a-judge prompts. The primary objective is to determine if APO can enhance the performance, strictness, and accuracy of the smaller, cheaper model (GPT-4o-mini) to match the baseline performance of the more expensive Anthropic model.


---

## 📚 Resources
* [ACI-BENCH dataset](https://arxiv.org/abs/2306.02022)
* [NOHARM2 Study](https://arxiv.org/abs/2512.01241)

## 💻 Technologies Used
* **Python** (Pandas, Matplotlib, Seaborn)
* **OpenAI API** (`gpt-4o-mini`)
* **Anthropic API** (`claude-sonnet-5`)
* **Google Gemini API** (`gemini-1.5-flash`)